In [1]:
# Loading the data 
from pathlib import Path

import kagglehub
import pandas as pd

COMPETITION_NAME = "50-007-machine-learning-may-2026"

competition_path = Path(kagglehub.competition_download(COMPETITION_NAME))

train_features_df = pd.read_csv(competition_path / "train_features.csv")

test_features_df = pd.read_csv(competition_path / "test_features.csv")

submission_df = pd.read_csv(competition_path / "sample_submission.csv")

C:\Users\siawx\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# import packages
import numpy as np
import os

from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

# separate features from id/label
feature_cols = []
for c in train_features_df.columns:
    if c not in ["id", "label"]:
        feature_cols.append(c)

# Training dataset
X_train = train_features_df[feature_cols]
Y_train = train_features_df["label"]

# Test dataset
X_test = test_features_df[feature_cols]
test_ids = test_features_df["id"]

In [ ]:
component_list = [2000, 1000, 500, 100]
results = {}

# Loop over component counts
for n_comp in component_list:
    print(f"\n Running PCA with n_components = {n_comp}")

    # create a PCA obj that will reduce down to n_comp dimensions
    pca = PCA(n_components = n_comp, random_state = 1)

    # learn the reduction from the trng data then apply it
    X_train_reduced = pca.fit_transform(X_train)

    # apply the same learned reduction to test data
    # use .transform & not .fit_transform cause test data must not influence PCA
    X_test_reduced = pca.transform(X_test)

    # create a fresh KNN classifier for this component count
    knn = KNeighborsClassifier(n_neighbors=2)

    # train KNN on reduced trng data + labels
    knn.fit(X_train_reduced, Y_train)

    # predict labels for reduced test data
    test_preds = knn.predict(X_test_reduced)

    # format predictions to expected format
    submission = pd.DataFrame({
        "id": test_ids,
        "label": test_preds
    })

    # make sure outputs directory exist
    os.makedirs("outputs", exist_ok = True)

    # save to csv file, 1 file/component
    filename = f"outputs/knn_pca_{n_comp}_predictions.csv"
    submission.to_csv(filename, index=False)
    print(f"Saved {filename} - Submit this to Kaggle to get its Macro F1 score")


 Running PCA with n_components = 2000
Saved knn_pca_2000_predictions.csv - Submit this to Kaggle to get its Macro F1 score

 Running PCA with n_components = 1000
Saved knn_pca_1000_predictions.csv - Submit this to Kaggle to get its Macro F1 score

 Running PCA with n_components = 500
Saved knn_pca_500_predictions.csv - Submit this to Kaggle to get its Macro F1 score

 Running PCA with n_components = 100
Saved knn_pca_100_predictions.csv - Submit this to Kaggle to get its Macro F1 score
